# First Agent

Contents:

- About The Agent We Will Build Throughout the Intensive
  - What is τ²-bench
  - RetailOps Agent - Task Overview
- Starting Agent Development
  - What an agent can do **with only an LLM**






# 1. About The Agent We Will Build Throughout the Intensive

## 1.1 What is τ²-bench


Recent benchmarks have been proposed to evaluate **LLM agents** that interact with users, tools, and real-world systems. In this intensive we will use τ²-bench.



**τ-bench** evaluates agents that must:

* converse with users
* use tools/APIs
* follow domain-specific rules

It simulates realistic customer-service scenarios such as retail and retail support.

**τ²-bench** extends this setup to a **dual-control environment**, where both the agent and the user can take actions via tools, making interactions more realistic.



### Papers

* **τ-bench: A Benchmark for Tool-Agent-User Interaction in Real-World Domains**
  [https://arxiv.org/abs/2406.12045](https://arxiv.org/abs/2406.12045)

* **τ²-Bench: Evaluating Conversational Agents in a Dual-Control Environment**
  [https://arxiv.org/abs/2506.07982](https://arxiv.org/abs/2506.07982)

### GitHub Repositories

* τ-bench: [https://github.com/sierra-research/tau-bench](https://github.com/sierra-research/tau-bench)
* τ²-bench: [https://github.com/sierra-research/tau2-bench](https://github.com/sierra-research/tau2-bench)






In [ ]:
# 1. Clone the repository
!git clone https://github.com/sierra-research/tau2-bench.git

# 2. Move into the directory
%cd tau2-bench

# 3. Install dependencies
!pip install -e .

[Generate](https://platform.openai.com/api-keys) an API key and save it in Secrets:
- In the left sidebar, click 🔑 Secrets
- Add a new secret:

```
Name: OPENAI_API_KEY
Value: sk-...
```


Or just paste it below.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
# !tau2 play

## 1.2. RetailOps Agent - Task Overview

We will implement an agent for the **retail support domain**.

The agent must:

* understand the user’s travel request
* gather missing information
* search products via APIs/tools
* respect return/refund/delivery rules and constraints
* complete or cancel an order safely

This scenario captures many core challenges of real-world AI agents: multi-turn dialogue, planning, tool use, and rule following.

In [ ]:
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, TypedDict

# Prefer retail domain; fallback keeps notebook runnable in environments
# where only the retail benchmark assets are available.
# Prefer retail domain; fallback keeps notebook runnable.
DOMAIN_DIR = Path("data/tau2/domains/retail")
if not DOMAIN_DIR.exists():
    DOMAIN_DIR = Path("data/tau2/domains/retail")
if not DOMAIN_DIR.exists():
    # Prefer retail domain; fallback keeps notebook runnable.
DOMAIN_DIR = Path("data/tau2/domains/retail")
if not DOMAIN_DIR.exists():
    DOMAIN_DIR = Path("data/tau2/domains/retail")

In [ ]:
from IPython.display import Markdown, display

display(Markdown((DOMAIN_DIR / "policy.md").read_text()))

In [ ]:
tasks_path = DOMAIN_DIR / "tasks.json"
with open(tasks_path, "r") as f:
    tasks_data = json.load(f)

print('Test case example')
print(json.dumps(tasks_data[0], indent=2))

In [ ]:
for i, task in enumerate(tasks_data[:3]):
    try:
        reason = task["user_scenario"]["instructions"]["reason_for_call"]
    except KeyError:
        reason = None

    print(f"{i}: {reason}\n")
    print("----------------")

## 1.3. How we will work on the agent

Step-by-step development: from a pure LLM baseline to a production-ready tool agent.

# 2. Starting Agent Development - What an agent can do **with only an LLM**

In this section we build the simplest possible agent: an agent that has only an LLM — no tools, no memory, no database, no RAG. This is our baseline, which we will extend in later lectures.

## 2.1. Prompt

In [ ]:
SYSTEM_PROMPT = """
You are a virtual retail support assistant.
IMPORTANT: In this lesson you have NO tools, NO database access, NO RAG.
You must be transparent about limitations:
- You cannot look up products, orders, customer profiles, delivery windows on a specific order, payment status, or real-time shipment status.
- You can only: explain the policy, ask clarifying questions, and outline what info is needed to proceed.
- Do NOT invent product details, order details, loyalty tier, prices, discounts, or statuses.
- Do NOT provide subjective recommendations.
"""

## 2.2 🧩 Lang* ecosystem for agents

In this course we will work with Lang* ecosystem.


What we need now:

- **LangChain** — a framework for building LLM applications.  
Provides abstractions for prompts, chains, tools, agents, memory, retrieval, etc. Use it for simple LLM apps

- **LangGraph** — an extension for building agents as state machines (graphs).  Gives precise control over multi-step workflows, tool use, loops, branching, and memory.  Use it for real agent with tools/workflow


what else is there
- **LangFlow** — visual builder (no/low-code UI) for LangChain pipelines and agents.  
- **LangSmith** — debugging, tracing, evaluation, monitoring for LLM apps.  
...



In [ ]:
!pip install -q langchain langgraph langchain-openai

## 2.3. LLM wrapper

`langchain_openai.ChatOpenAI` — a wrapper around OpenAI chat models. It handles API calls, message formatting, tokenization, and response parsing. We interact with the model as a structured chat rather than raw text.

In [ ]:
from langchain_openai import ChatOpenAI

MODEL = "gpt-5-nano"

llm = ChatOpenAI(
    model=MODEL,
    temperature=0.2,
)


## 2.4. Message wrappers

`langchain_core.messages` — defines message types used by chat models:
- SystemMessage — instructions for the model (policy, role)
- HumanMessage — user input
- AIMessage — model output
- BaseMessage — parent type for all messages


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage

## 2.5. LangGraph details

Instead of a simple pipeline (user → LLM → answer), we define a graph where `state` flows through `nodes`. This is how production agents are built, because it allows adding tools, memory, retrieval, and multi-step reasoning later.




### 2.5.1 Agent State

We define the agent’s state as a typed structure containing the conversation history. The state stores the messages exchanged so far. More advanced agents may also store tool outputs, retrieved documents, user profiles, plans, or intermediate reasoning.


In [ ]:
class AgentState(TypedDict):
    messages: List[BaseMessage]

### 2.5.2. Nodes

A node is a function that:
1. Receives the current state  
2. Performs computation  
3. Returns an updated state  

Our LLM node:
- Prepends the system prompt (agent policy)
- Calls the LLM with the conversation history
- Appends the model’s reply back into the state

This is how the conversation grows over time.

In [ ]:
def llm_node(state: AgentState) -> AgentState:
    msgs = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    resp = llm.invoke(msgs)
    return {"messages": state["messages"] + [resp]}


### 2.5.3. Agent as a StateGraph

We construct a graph where:
- nodes are functions  
- edges define transitions  
- state flows between nodes  

Right now the graph is:

START → LLM → END

So this is a single-step agent. Later we will extend it to multi-step flows like:

LLM → Tool → LLM → Memory → ...


#### 1) Entry point and edges

The entry point defines where execution starts (the LLM node). The edge to END means that after one LLM step, the agent stops.

In [ ]:

from langgraph.graph import StateGraph, END


graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.set_entry_point("llm")
graph.add_edge("llm", END)
graph

#### 2) Compile

Compilation transforms the graph definition into an executable program that can be invoked with an initial state.

In [ ]:
app = graph.compile()
app


#### 3) Running one interaction

The helper function:
1. Creates the initial state with a user message  
2. Runs the graph  
3. Returns the model’s reply  

In [ ]:

def ask(user_text: str):
    state = {"messages": [HumanMessage(content=user_text)]}
    out = app.invoke(state)
    return out["messages"][-1].content


# ✅ 2 questions the LLM-only agent CAN answer (policy/constraints reasoning)
CAN_ANSWER = [
    "1) According to the policy, when do you need explicit user confirmation before taking an action?",
    "2) Explain the checked bag allowance rules for regular vs silver vs gold members across cabin classes.",
]

# ❌ 2 questions from your benchmark examples that the LLM-only agent CANNOT solve without tools/DB
CANNOT_ANSWER = [
    "1) Cancel reservation EHGLP3.",
    "2) I’m contacting to complain about my delayed product HAT045 from PHX to SEA. What compensation can you issue?",
]

print("=== LLM-only agent demo ===\n")

print("---- CAN answer (policy-based) ----")
for q in CAN_ANSWER:
    print(f"\nUSER: {q}")
    print(f"AGENT: {ask(q)}")

print("\n\n---- CANNOT answer (needs tools/DB) ----")
for q in CANNOT_ANSWER:
    print(f"\nUSER: {q}")
    print(f"AGENT: {ask(q)}")


### 🎯 Key takeaway

This agent:
- has an LLM  
- has no tools  
- has no database  
- has no external knowledge  
- cannot perform actions  

It can only:
- follow the policy  
- explain rules  
- ask clarifying questions  